# Capstone — ranking pages for refresh, validated on a month it never saw

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkusmade/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (locked in ML-07).
**Question:** *of a client portfolio's pages, which should an editor review first next month — and does a
learned ranking beat a transparent rule on a month it has never seen?*

This notebook is the evidence behind the deployed paper. Every number on the page comes from a cell
below; the page quotes this run, and `work/outputs/capstone_metrics.json` is the receipt.

| Section | What it holds |
|---|---|
| 1 | the decision, the unit, the action, the cost of a wrong call |
| 2 | data: release, tables, windows, what is excluded and why |
| 3 | methodology: eight features, the label, the frozen baseline, grouped + future validation, leakage checks |
| 4 | results: grouped vs random, then the future-month test, then who the queues contain |
| 5 | limitations, measured |
| 6 | the ranked action playbook (with an as-of-the-decision queue) |
| 7 | the charts the page embeds |
| 8–9 | reproducibility and the data credit |

**Data:** `FlyRank/internship-warehouse`, build `flyrank_pseudonymized_warehouse_release_v20260703`,
read with DuckDB over `hf://`. Token: local `.env` (gitignored) or Colab Secret `HF_TOKEN` — never in a cell.

**The sealed month is now opened, once.** ML-04 and ML-07 kept 2026-06 unread. It is used here for the
single thing it was saved for: the **label of the future test window** (features 2026-05, outcome 2026-06).
It is not used for training, feature selection, or threshold tuning. The training window is
2026-03 → 2026-04.

In [1]:
# --- Setup: works in Colab (clones this repo) and locally (reads .env, walks up to the repo root) ---
import json
import os
import subprocess
import sys
import time
from datetime import date, datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mayurkusmade/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not Path("data/raw").is_dir() and Path.cwd().parent != Path.cwd():
        os.chdir("..")


def hf_token():
    token = os.environ.get("HF_TOKEN")
    if not token and Path(".env").exists():
        for line in Path(".env").read_text().splitlines():
            if line.strip().startswith("HF_TOKEN="):
                token = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token:
        import getpass
        token = getpass.getpass("Hugging Face READ token (hf_...): ")
    return token


import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token()}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
BUILD = "flyrank_pseudonymized_warehouse_release_v20260703"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
OUT = Path("work/outputs")
FIG = Path("paper/img")
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SEED = 42
GATE = 500                     # an editor slot must be worth something: >= 500 monthly impressions
TRAIN_FEATURES, TRAIN_LABEL = "2026-03", "2026-04"
TEST_FEATURES, TEST_LABEL = "2026-05", "2026-06"      # the sealed month, used once
DECISION_TRAIN, DECISION_TEST = date(2026, 3, 31), date(2026, 5, 31)
assert TRAIN_LABEL < TEST_FEATURES, "the test feature window must be after the training label window"
assert TEST_FEATURES < TEST_LABEL, "features must close before their own label window"

print(f"release   : {BUILD}")
print(f"train     : features {TRAIN_FEATURES} -> label {TRAIN_LABEL}")
print(f"future    : features {TEST_FEATURES} -> label {TEST_LABEL}  (June: sealed until now, opened once)")
print(f"seed      : {SEED}     gate: >= {GATE} impressions")

release   : flyrank_pseudonymized_warehouse_release_v20260703
train     : features 2026-03 -> label 2026-04
future    : features 2026-05 -> label 2026-06  (June: sealed until now, opened once)
seed      : 42     gate: >= 500 impressions


## 1. Question and the decision it supports

An editor at a content agency has roughly **50 review slots a week** and handles several clients. Each
slot costs a day of writing, editing and QA, so choosing badly is expensive twice over: the refresh
does not happen, and a page that did need it does not get one.

- **Unit of analysis:** one content item for one client, summarised over one calendar month.
- **Output:** a ranked queue with a score, a reason code and one action per item.
- **Human action:** `check_indexing`, `refresh_content`, or `monitor`.
- **Cost of a wrong call:** a wasted slot, or a decline that runs a month longer than it had to.

The decision is a ranking problem, so the metric has to be **precision@K** (of the top K, how many were
right), always next to the **base rate** — the share that would be "right" with no model at all.

In [2]:
decision = pd.DataFrame([
    ("unit", "one content item x one client, summarised over one month"),
    ("output", "ranked queue: score + reason code + action"),
    ("editor capacity", "~50 slots per week  ->  K = 50"),
    ("actions", "check_indexing | refresh_content | monitor"),
    ("primary metric", "precision@50 on a future month, with the base rate beside it"),
    ("secondary metric", "ROC-AUC (whole-population ranking), precision@100/500"),
    ("cost of a miss", "a wasted slot; a decline that runs a month longer than needed"),
], columns=["decision element", "value"])
display(decision)

,decision element,value
0,unit,"one content item x one client, summarised over..."
1,output,ranked queue: score + reason code + action
2,editor capacity,~50 slots per week -> K = 50
3,actions,check_indexing | refresh_content | monitor
4,primary metric,"precision@50 on a future month, with the base ..."
5,secondary metric,"ROC-AUC (whole-population ranking), precision@..."
6,cost of a miss,a wasted slot; a decline that runs a month lon...


## 2. Data

**Release:** `FlyRank/internship-warehouse` (build `flyrank_pseudonymized_warehouse_release_v20260703`),
tables `fact_content_daily_performance` and `dim_content`, read with DuckDB over `hf://`.

| Window | Feature month | Label month | Rows | Used for |
|---|---|---|---|---|
| Train | 2026-03 | 2026-04 | 100,893 items / 43 clients | fitting, model selection |
| Future test | 2026-05 | 2026-06 | 111,075 items / 48 clients | evaluation only, once |

**Excluded, on purpose:**

- `fact_content_query_90d` — its window (2026-04-02 → 2026-06-30) *contains* the label month. ML-04
  measured it. Nothing from it is used here.
- Every current-state field in `dim_content` except `content_created_date`: `content_updated_date`,
  `last_optimized_date`, `is_published`, `is_deleted`, `word_count` all describe the snapshot after
  the decision and would import 2026-04+ activity into a March decision. ML-07 measured the censoring
  (82.1% of items updated after the decision date).
- Rows with `gsc_data_available IS NOT TRUE`, and items under 100 monthly impressions.
- Product flags (`health_score`, `priority_score`, `action_type`) — they do not exist in this release,
  and if they did they would be a baseline to beat, never an input.

In [3]:
# --- Build (or load) both windows. One heavy scan each: 2 month partitions per window. ---
MIN_IMP = 100


def month_ref(mm):
    return f"read_parquet('{REL}/fact_content_daily_performance/month={mm}/data_0.parquet')"


def build_window(feat_month, label_month, split_day, tag):
    cache = OUT / f"capstone_frame_{tag}.parquet"
    if cache.exists():
        df = pd.read_parquet(cache)
        print(f"{tag}: cache {cache.name} ({len(df):,} rows)")
        return df
    sql = f'''
        WITH feat AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) AS imp,
                   SUM(gsc_clicks)      AS clk,
                   AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos,
                   COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_imp,
                   SUM(CASE WHEN report_date >= DATE '{split_day}' THEN gsc_impressions ELSE 0 END) AS imp_h2,
                   SUM(CASE WHEN report_date <  DATE '{split_day}' THEN gsc_impressions ELSE 0 END) AS imp_h1
            FROM {month_ref(feat_month)}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
            HAVING SUM(gsc_impressions) >= {MIN_IMP}
        ),
        lab AS (
            SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next
            FROM {month_ref(label_month)}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT f.*, l.imp_next
        FROM feat f LEFT JOIN lab l USING (client_hash_id, content_hash_id)
    '''
    for attempt in range(1, 4):
        try:
            df = con.sql(sql).df()
            break
        except Exception as exc:
            print(f"  scan attempt {attempt} failed ({type(exc).__name__}); retrying")
            time.sleep(10 * attempt)
    else:
        raise RuntimeError(f"could not build the {tag} window")
    df.to_parquet(cache, index=False)
    print(f"{tag}: live scan ({len(df):,} rows)")
    return df


train_raw = build_window(TRAIN_FEATURES, TRAIN_LABEL, "2026-03-16", "mar")
test_raw = build_window(TEST_FEATURES, TEST_LABEL, "2026-05-16", "may")

# content age is the only dim_content field used: created_date cannot change after the fact.
ages = con.sql(f'''
    SELECT content_hash_id,
           DATE '{DECISION_TRAIN}' - content_created_date AS age_train,
           DATE '{DECISION_TEST}'  - content_created_date AS age_test
    FROM {DIM_CONTENT}
''').df()


def prep(df, age_col):
    df = df.merge(ages[["content_hash_id", age_col]], on="content_hash_id", how="left").copy()
    df["momentum"] = np.where(df.imp_h1 > 0, df.imp_h2 / df.imp_h1, np.nan)
    df["ctr"] = 100 * df.clk / df.imp
    df["has_h1"] = (df.imp_h1 == 0).astype(int)          # momentum undefined: a flag, not a fill
    df["pos_missing"] = df.pos.isna().astype(int)
    df["pos"] = df.pos.fillna(df.pos.median())
    df["log_imp"] = np.log1p(df.imp)
    df["momentum_f"] = df.momentum.fillna(1.0).clip(0, 3)
    df["age_days"] = df[age_col].fillna(df[age_col].median())
    df = df.dropna(subset=["imp_next"]).copy()
    df["label"] = (df.imp_next < 0.8 * df.imp).astype(int)
    return df


train = prep(train_raw, "age_train")
test = prep(test_raw, "age_test")
FEATURES = ["log_imp", "days_with_imp", "momentum_f", "has_h1", "ctr", "pos", "pos_missing", "age_days"]

train_gated = train[train.imp >= GATE]
test_gated = test[test.imp >= GATE]

summary = pd.DataFrame([
    ("train", len(train), train.client_hash_id.nunique(), round(train.label.mean(), 3), len(train_gated)),
    ("future test", len(test), test.client_hash_id.nunique(), round(test.label.mean(), 3), len(test_gated)),
], columns=["window", "items", "clients", "decline base rate", f"items >= {GATE} imp"])
display(summary)
print(f"items dropped for having no label-month row: train {len(train_raw) - len(train):,}, "
      f"test {len(test_raw) - len(test):,}  (traffic to zero and item removal are indistinguishable)")

mar: cache capstone_frame_mar.parquet (101,441 rows)
may: cache capstone_frame_may.parquet (112,309 rows)


,window,items,clients,decline base rate,items >= 500 imp
0,train,100893,43,0.515,61846
1,future test,111075,48,0.656,59874


items dropped for having no label-month row: train 548, test 1,234  (traffic to zero and item removal are indistinguishable)


## 3. Methodology

**Eight features, each knowable at the decision date, all measured inside the feature month:**

| Feature | How it is computed | Knowable at the decision moment because… |
|---|---|---|
| `log_imp` | log1p of the month's impressions | the month is closed; the editor sees the same number |
| `days_with_imp` | days with > 0 impressions | count of days already observed |
| `momentum_f` | 2nd half ÷ 1st half of the month, clipped at 3 | both halves are in the past |
| `has_h1` | 1 when the first half had zero impressions | flags "momentum undefined", not zero |
| `ctr` | clicks ÷ impressions × 100 | same closed month |
| `pos` | mean position, `0` treated as "no data" | same closed month |
| `pos_missing` | 1 when position was never reported | a flag, so a fill cannot pass as a measurement |
| `age_days` | decision date − `content_created_date` | the creation date is immutable |

**Label (the target):** `label = 1` when the next month's impressions fall **at least 20%** below the
feature month's. It is a *defined proxy over observed outcomes* — a chosen band, not a natural fact.

**Baseline (frozen from ML-07, not re-tuned):** eligible when `imp >= 500`, `imp_h1 > 0` and
`momentum < 0.8`; `score = imp × (1 − momentum)`; one reason code, one action. It appears in every
table below, computed in this run on the same rows.

**Validation design — two questions, two splits:**

1. **Will it work on a client it has never seen?** 5 × 20% **grouped by `client_hash_id`**, on the
   training month. A random split of the same rows is reported beside it; the gap is the memorisation.
2. **Will it work next month?** Fit on all of March, predict May, check against June. One shot.

**Method choice:** logistic regression (readable, 8 coefficients) is the model. Random forest and
gradient boosting are run as challengers. The challenger that wins the random split is *not*
automatically the choice — section 4 is where that gets decided.

In [4]:
# --- Leakage checks first: a model is only worth training on a frame that passes them ---
checks = {
    "feature columns are all in the frame": all(c in train.columns for c in FEATURES),
    "no label-month column in FEATURES": not any(("next" in c or "label" in c or c.startswith("apr")) for c in FEATURES),
    "train feature month < train label month": TRAIN_FEATURES < TRAIN_LABEL,
    "train label month < test feature month": TRAIN_LABEL < TEST_FEATURES,
    "test feature month < test label month": TEST_FEATURES < TEST_LABEL,
    "no dim_content snapshot fields used": not any(c in FEATURES for c in
        ("content_updated_date", "last_optimized_date", "is_published", "is_deleted", "word_count")),
    "momentum is NaN exactly where has_h1 == 1": bool(train.loc[train.has_h1 == 1, "momentum"].isna().all()
                                                       and train.loc[train.has_h1 == 0, "momentum"].notna().all()),
    "labels are binary": set(train.label.unique()) == {0, 1} and set(test.label.unique()) == {0, 1},
}
for k, v in checks.items():
    print(f"  {k:46} {v}")
assert all(checks.values()), "a leakage check failed"

# The score must not move when every outcome column is deleted: rebuild the feature matrix without them.
scratch = train.drop(columns=["imp_next", "label"])
rebuilt = scratch[FEATURES].copy()
assert np.allclose(rebuilt.fillna(0).values, train[FEATURES].fillna(0).values), "features touch an outcome column"
print("\nfeature matrix is identical with all outcome columns deleted -> no label contact")

  feature columns are all in the frame           True
  no label-month column in FEATURES              True
  train feature month < train label month        True
  train label month < test feature month         True
  test feature month < test label month          True
  no dim_content snapshot fields used            True
  momentum is NaN exactly where has_h1 == 1      True
  labels are binary                              True

feature matrix is identical with all outcome columns deleted -> no label contact


### 3.1 Model selection — grouped vs random, on the training month

Four candidates, same rows, same folds, same K. `rule` is the ML-07 baseline.

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def make_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED))


def make_rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=50, max_features="sqrt",
                                  n_jobs=-1, random_state=SEED, class_weight="balanced_subsample")


def make_gbm():
    return HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.06,
                                          random_state=SEED, early_stopping=False)


def rule_score(df):
    ok = (df.imp >= GATE) & (df.imp_h1 > 0) & (df.momentum < 0.8)
    return np.where(ok, df.imp * (1 - df.momentum), 0.0)


def precision_at_k(y, s, k):
    order = np.argsort(-np.asarray(s))
    return float(np.asarray(y)[order[:k]].mean())


CANDIDATES = {"logit": make_logit, "rf": make_rf, "gbm": make_gbm}


def evaluate_split(scheme, splitter):
    rows = []
    for name, factory in CANDIDATES.items():
        aucs, p50 = [], []
        it = (splitter.split(train, train.label, groups=train.client_hash_id) if scheme == "grouped"
              else splitter.split(train, train.label))
        for fit_i, val_i in it:
            model = factory().fit(train.iloc[fit_i][FEATURES], train.iloc[fit_i].label)
            proba = model.predict_proba(train.iloc[val_i][FEATURES])[:, 1]
            aucs.append(roc_auc_score(train.iloc[val_i].label, proba))
            p50.append(precision_at_k(train.iloc[val_i].label, proba, 50))
        rows.append({"model": name, "auc": np.mean(aucs), "p50": np.mean(p50)})
    aucs, p50 = [], []
    it = (splitter.split(train, train.label, groups=train.client_hash_id) if scheme == "grouped"
          else splitter.split(train, train.label))
    for fit_i, val_i in it:
        s = rule_score(train.iloc[val_i])
        aucs.append(roc_auc_score(train.iloc[val_i].label, s))
        p50.append(precision_at_k(train.iloc[val_i].label, s, 50))
    rows.append({"model": "rule", "auc": np.mean(aucs), "p50": np.mean(p50)})
    for r in rows:
        r["scheme"] = scheme
    return rows


grouped = evaluate_split("grouped", GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=SEED))
random_ = evaluate_split("random", StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=SEED))
selection = (pd.DataFrame(grouped + random_)
             .pivot(index="model", columns="scheme", values=["p50", "auc"])
             .reindex(["rule", "logit", "rf", "gbm"]))
selection.columns = [f"{a}_{b}" for a, b in selection.columns]
selection = selection[["p50_grouped", "p50_random", "auc_grouped", "auc_random"]].round(3)
selection["p50_gap_random_minus_grouped"] = (selection.p50_random - selection.p50_grouped).round(3)
display(selection)
print(f"base rate in the training month: {train.label.mean():.3f}   "
      f"fold size ~{(0.2 * len(train)):,.0f} rows")

,p50_grouped,p50_random,auc_grouped,auc_random,p50_gap_random_minus_grouped
model,,,,,
rule,0.816,0.876,0.550,0.570,0.060
logit,0.844,0.900,0.677,0.713,0.056
rf,0.704,0.972,0.655,0.776,0.268
gbm,0.836,0.984,0.656,0.777,0.148


base rate in the training month: 0.515   fold size ~20,179 rows


**What the split does to the story.** On a random split, gradient boosting looks excellent (P@50 0.984) and
random forest nearly as good (0.972) — while the rule manages 0.876. Group the folds by client and the order
inverts: boosting drops to 0.836, forest to **0.704**, and logistic regression holds at **0.844** with the
best grouped AUC (0.677). The flexible models were partly memorising *clients*, not pages: with 43 clients
and one of them large, a random split lets a tree recognise a client and score its pages from its peers.

That is why the model I carry forward is the boring one — **logistic regression**, eight coefficients,
stable across both splits (0.900 random → 0.844 grouped, a 0.056 gap versus the forest's 0.268).

## 4. Results — the frozen model on a month it never saw

Trained once on 2026-03 (43 clients), applied unchanged to 2026-05 (48 clients), scored against 2026-06.
Nothing is re-fitted or re-tuned on the test window. The gated population (≥ 500 impressions) is the one
an editor would actually work from; the whole-population numbers are shown so the gate's effect is visible.

In [6]:
results = []
fitted = {}
for name, factory in CANDIDATES.items():
    model = factory().fit(train[FEATURES], train.label)
    fitted[name] = model
    results.append({
        "model": name,
        "p@50_all": precision_at_k(test.label, model.predict_proba(test[FEATURES])[:, 1], 50),
        "auc_all": roc_auc_score(test.label, model.predict_proba(test[FEATURES])[:, 1]),
        "p@50_gated": precision_at_k(test_gated.label, model.predict_proba(test_gated[FEATURES])[:, 1], 50),
        "auc_gated": roc_auc_score(test_gated.label, model.predict_proba(test_gated[FEATURES])[:, 1]),
    })
rs_all, rs_gated = rule_score(test), rule_score(test_gated)
results.append({"model": "rule", "p@50_all": precision_at_k(test.label, rs_all, 50),
                "auc_all": roc_auc_score(test.label, rs_all),
                "p@50_gated": precision_at_k(test_gated.label, rs_gated, 50),
                "auc_gated": roc_auc_score(test_gated.label, rs_gated)})
future = pd.DataFrame(results).set_index("model").reindex(["rule", "logit", "rf", "gbm"]).round(3)
display(future)

test_proba = fitted["logit"].predict_proba(test[FEATURES])[:, 1]
test_gated_proba = fitted["logit"].predict_proba(test_gated[FEATURES])[:, 1]

# bootstrap CI on P@50: 50 rows is a small sample, so the interval matters more than the point
rng = np.random.default_rng(SEED)


def p50_ci(y, s, k=50, draws=2000):
    hits = np.asarray(y)[np.argsort(-np.asarray(s))[:k]]
    vals = [hits[rng.integers(0, k, k)].mean() for _ in range(draws)]
    return np.percentile(vals, [2.5, 97.5])


ci_rows = []
for name, s_vec in [("rule", rs_gated),
                    ("logit", test_gated_proba),
                    ("rf", fitted["rf"].predict_proba(test_gated[FEATURES])[:, 1]),
                    ("gbm", fitted["gbm"].predict_proba(test_gated[FEATURES])[:, 1])]:
    p50 = precision_at_k(test_gated.label, s_vec, 50)
    lo, hi = p50_ci(test_gated.label, s_vec)
    ci_rows.append({"ranking": name, "p@50": p50, "ci_low": lo, "ci_high": hi})
ci = pd.DataFrame(ci_rows).round(3)
display(ci)

print(f"population sizes  : whole {len(test):,} (base {test.label.mean():.3f}) | "
      f"gated {len(test_gated):,} (base {test_gated.label.mean():.3f})")
print(f"hit counts at 50  : rule {int(round(precision_at_k(test_gated.label, rs_gated, 50) * 50))}/50, "
      f"logit {int(round(precision_at_k(test_gated.label, test_gated_proba, 50) * 50))}/50")

# who is in each queue? coverage matters as much as precision
coverage = []
for name, s in [("rule", rs_gated), ("logit (p)", test_gated_proba)]:
    top = test_gated.assign(score=s).nlargest(50, "score")
    coverage.append({"ranking": name, "clients_in_top50": top.client_hash_id.nunique(),
                     "largest_client_share": int(top.client_hash_id.value_counts().iloc[0]),
                     "median_impressions": int(top.imp.median())})
for weighting in ("log", "sqrt", "raw"):
    factor = {"log": np.log1p(test_gated.imp), "sqrt": np.sqrt(test_gated.imp), "raw": test_gated.imp}[weighting]
    s = test_gated_proba * factor
    top = test_gated.assign(score=s).nlargest(50, "score")
    coverage.append({"ranking": f"logit x {weighting}(imp)",
                     "clients_in_top50": top.client_hash_id.nunique(),
                     "largest_client_share": int(top.client_hash_id.value_counts().iloc[0]),
                     "median_impressions": int(top.imp.median())})
for row in coverage:
    row["p@50"] = round(precision_at_k(test_gated.label, (
        rs_gated if row["ranking"] == "rule" else
        test_gated_proba if row["ranking"] == "logit (p)" else
        test_gated_proba * (np.log1p(test_gated.imp) if row["ranking"].endswith("log(imp)")
                            else np.sqrt(test_gated.imp) if row["ranking"].endswith("sqrt(imp)")
                            else test_gated.imp)), 50), 3)
display(pd.DataFrame(coverage))

# precision@K curve for the page's chart
ks = [25, 50, 100, 200, 500, 1000]
curve = pd.DataFrame({"k": ks,
                      "rule": [precision_at_k(test_gated.label, rs_gated, k) for k in ks],
                      "logit": [precision_at_k(test_gated.label, test_gated_proba, k) for k in ks],
                      "logit x log(imp)": [precision_at_k(test_gated.label, test_gated_proba * np.log1p(test_gated.imp), k)
                                           for k in ks]}).round(3)
display(curve)

,p@50_all,auc_all,p@50_gated,auc_gated
model,,,,
rule,0.94,0.563,0.94,0.615
logit,0.90,0.698,0.98,0.705
rf,0.86,0.689,0.88,0.696
gbm,0.90,0.667,0.90,0.667


,ranking,p@50,ci_low,ci_high
0,rule,0.94,0.86,1.00
1,logit,0.98,0.94,1.00
2,rf,0.88,0.78,0.96
3,gbm,0.90,0.82,0.98


population sizes  : whole 111,075 (base 0.656) | gated 59,874 (base 0.665)
hit counts at 50  : rule 47/50, logit 49/50


,ranking,clients_in_top50,largest_client_share,median_impressions,p@50
0,rule,11,16,55348,0.94
1,logit (p),20,9,674,0.98
2,logit x log(imp),14,14,29009,0.98
3,logit x sqrt(imp),10,23,118895,0.76
4,logit x raw(imp),12,21,140845,0.70


,k,rule,logit,logit x log(imp)
0,25,0.960,1.000,0.960
1,50,0.940,0.980,0.980
2,100,0.940,0.970,0.980
3,200,0.935,0.925,0.965
4,500,0.920,0.934,0.924
5,1000,0.943,0.918,0.933


**The honest result, in three sentences.**

1. **Inside the deployment population the learned ranking is the best one at the editor's K:** 0.980 P@50
   (49 of 50) for logistic regression against 0.940 (47 of 50) for the rule — a **two-page difference**,
   with bootstrap intervals (0.94–1.00 vs 0.86–1.00) that overlap. It is an improvement, not a transformation.
2. **The baseline is not humiliated**, and that is the interesting part. A three-condition rule with no
   fitting reaches 47/50 on a month six weeks after its thresholds were set, at 11 clients in the top 50.
   Any model that claims to beat it needs this table, not a leaderboard.
3. **The two rankings reach nearly the same precision on different pages.** At 50 slots the choice is not
   "which is more accurate" but "which portfolio you want to cover": the rule's top 50 has a median of
   **55,348** impressions and spans 11 clients; the model's spans **20 clients** with a median of 674.

In [7]:
# --- What the model leans on, in plain numbers ---
coef = pd.Series(fitted["logit"][-1].coef_[0], index=FEATURES).sort_values()
coef_table = coef.to_frame("standardised coefficient").reset_index().rename(columns={"index": "feature"})
display(coef_table.round(3))
print("negative = lowers next month's decline risk; every feature is a feature-month measurement")

,feature,standardised coefficient
0,momentum_f,-0.595
1,has_h1,-0.424
2,ctr,-0.370
3,log_imp,-0.203
4,age_days,-0.083
5,pos,-0.083
6,pos_missing,0.000
7,days_with_imp,0.057


negative = lowers next month's decline risk; every feature is a feature-month measurement


Read the coefficients as direction, not as effect sizes on the outcome: **momentum** carries by far the
largest weight (−0.595 standardised) — a page that ended the month weaker is more likely to fall again —
followed by **`has_h1`** (−0.424, pages whose traffic only started in the second half are the safest group,
exactly as ML-07 measured) and **CTR** (−0.370). `age_days` is −0.083: older content is *very slightly*
less likely to decline, which is the third independent confirmation that staleness is not the signal the
refresh heuristic assumes. `pos_missing` sits at 0.000 because it separates no rows.

## 5. Limitations

Written before a reader writes them for me:

1. **One training month, one test month.** The model saw March only (43 clients) and was scored on May
   only. A model trained over more months would see seasonality that this one cannot.
2. **The label's base rate moves a lot between months:** 0.515 in March → 0.656 in May→June, and 0.665 in
   the gated population. Precision@K is only readable next to that number, and a single test month gives
   one draw, not a distribution.
3. **The ±20% band is a chosen proxy.** A client whose whole inventory fell labels nearly everything; a
   healthy client labels almost nothing.
4. **The test month's clients overlap the training clients' *portfolio*, not their rows** — 48 test clients
   against 43 training clients. Grouped CV answers "unseen client"; the future test answers "unseen month".
   Neither answers both at once.
5. **K=50 is small.** A difference of two pages moves P@50 by 0.04. The bootstrap intervals overlap, and I
   report them instead of ranking models by a coin-flip.
6. **The queue is not causal.** Nothing here shows that refreshing a page recovers traffic. It ranks pages
   by an observed past→future pattern; the counterfactual is unmeasured.
7. **Items with no next-month row are dropped** (548 in training, on the March frame): "traffic fell to
   zero" and "the item was removed" look identical, and the largest declines live in that gap.
8. **One top-ranked failure mode is structural, not editorial.** In ML-07 the single highest-scoring page
   dropped from 28.9k to 2 impressions in four days — a URL/indexing/tracking break. The playbook's
   `check_indexing` action exists because of it, and it is a rule of thumb, not a validated classifier.
9. **No client-identifying data is used or shown.** IDs are pseudonyms, truncated in every table.

In [8]:
# --- the limitations that can be measured, measured ---
lim = pd.DataFrame([
    ("train base rate", round(train.label.mean(), 3)),
    ("test base rate (whole)", round(test.label.mean(), 3)),
    ("test base rate (gated)", round(test_gated.label.mean(), 3)),
    ("items dropped for no label row (train)", len(train_raw) - len(train)),
    ("clients: train / test", f"{train.client_hash_id.nunique()} / {test.client_hash_id.nunique()}"),
    ("P@50 gap, rule vs logit (gated)", round(
        precision_at_k(test_gated.label, test_gated_proba, 50)
        - precision_at_k(test_gated.label, rs_gated, 50), 3)),
    ("that gap in pages", int(round((precision_at_k(test_gated.label, test_gated_proba, 50)
                                     - precision_at_k(test_gated.label, rs_gated, 50)) * 50))),
], columns=["measured limitation", "value"])
display(lim)

,measured limitation,value
0,train base rate,0.515
1,test base rate (whole),0.656
2,test base rate (gated),0.665
3,items dropped for no label row (train),548
4,clients: train / test,43 / 48
5,"P@50 gap, rule vs logit (gated)",0.04
6,that gap in pages,2


## 6. Ranked recommendations — the action playbook

**Scoring for the queue:** `final_score = P(decline) × log(1 + impressions)`.

Why the second factor: pure probability puts small, noisy pages at the top — a 20% wobble on 600
impressions is easier to "predict" than a real decline on 60,000. Weighting by log size keeps the head of
the queue precise (0.98 P@50, unchanged) while raising the median impressions of queued pages from
**674 to 29,009** — 40× more traffic at stake per slot. Weighting by raw impressions instead lets size
dominate risk and costs precision (see the table below).

**Actions, assigned from the feature month alone:**

| Action | Condition | What the editor does |
|---|---|---|
| `check_indexing` | momentum < 0.10 — traffic stopped mid-month | verify canonical, redirects, robots/noindex, analytics before touching copy |
| `refresh_content` | momentum < 0.8 and ≥ 500 impressions | rewrite/update the page, then measure against the baseline |
| `monitor` | everything else the model still ranks | watch one more month; no slot yet |

The queue below is built from the **May feature month** — the test window — so every recommendation is
backtested: the `imp_next` column shows what June actually did, and it is shown for review only, never
used to compute the score.

In [9]:
# --- the playbook queue, built exactly as a deployment would build it ---
queue = test_gated.copy()
queue["p_decline"] = test_gated_proba
queue["score"] = queue.p_decline * np.log1p(queue.imp)
queue = queue.sort_values(["score", "content_hash_id"], ascending=[False, True]).head(50).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)


def assign_action(row):
    if row.momentum < 0.10:
        return "check_indexing", "structural_break"
    if row.momentum < 0.8:
        return "refresh_content", "visible_slip"
    return "monitor", "model_only"


queue[["action", "reason_code"]] = [assign_action(r) for r in queue.itertuples()]

cols = ["rank", "client_hash_id", "content_hash_id", "score", "p_decline", "action", "reason_code",
        "imp", "imp_h1", "imp_h2", "momentum", "ctr", "pos", "days_with_imp", "age_days"]
queue[cols].round(4).to_csv(OUT / "capstone_action_queue.csv", index=False)

top10 = queue.head(10).assign(client_key=lambda d: d.client_hash_id.str[:14],
                              content_key=lambda d: d.content_hash_id.str[:14],
                              june_pct=lambda d: ((d.imp_next - d.imp) / d.imp * 100).round(1))
display(top10[["rank", "client_key", "content_key", "score", "action", "imp", "momentum", "ctr",
               "pos", "june_pct"]].round(3))

print(queue.groupby("action").label.agg(["size", "mean"]).round(3).to_string())
print(f"\nqueue: {len(queue)} rows, {queue.client_hash_id.nunique()} clients, "
      f"median impressions {int(queue.imp.median()):,}")
print(f"top-10 hits in June: {int(queue.head(10).label.sum())}/10   top-50: {int(queue.label.sum())}/50")
print(f"queue written to {OUT / 'capstone_action_queue.csv'} (gitignored: regenerated on every run)")

,rank,client_key,content_key,score,action,imp,momentum,ctr,pos,june_pct
0,1,client_0fa64a1,content_11bf4c,7.948,check_indexing,88175.0,0.016,0.001,13.473,21.6
1,2,client_3f0ce4d,content_f79972,7.551,refresh_content,75264.0,0.142,0.036,9.233,-92.3
2,3,client_62f4a7e,content_a14cf3,7.551,check_indexing,40880.0,0.039,0.007,7.394,-72.5
3,4,client_73cda7b,content_58fa49,7.523,refresh_content,75272.0,0.171,0.035,7.042,-73.1
4,5,client_a80fca3,content_7918f4,7.388,refresh_content,73849.0,0.304,0.005,8.066,-25.8
5,6,client_e5c2aa2,content_72524c,7.370,refresh_content,67712.0,0.176,0.096,7.741,-56.5
6,7,client_e547b89,content_4ab17d,7.368,check_indexing,29530.0,0.014,0.003,9.950,-83.0
7,8,client_62f4a7e,content_3961d5,7.306,refresh_content,45102.0,0.195,0.007,8.630,-88.9
8,9,client_62f4a7e,content_762945,7.288,refresh_content,53772.0,0.129,0.028,9.844,-96.6
9,10,client_73cda7b,content_60e0c9,7.268,refresh_content,46086.0,0.114,0.072,3.675,-47.7


                 size   mean
action                      
check_indexing     22  0.955
refresh_content    28  1.000

queue: 50 rows, 14 clients, median impressions 29,009
top-10 hits in June: 9/10   top-50: 49/50
queue written to work\outputs\capstone_action_queue.csv (gitignored: regenerated on every run)


**How an editor uses this tomorrow.** Work down the queue in order, but read the action first: a
`check_indexing` row is a 15-minute technical check, not a rewrite; a `refresh_content` row is the real
writing slot; `monitor` means the score is high but no pattern is clear, so one more month of data is
cheaper than a guess. After four weeks, compare the refreshed pages' impressions against the
`visible_slip` baseline rate — that comparison is what would eventually turn this ranking into evidence
about refreshes, and it is not something this dataset can do on its own.

## 7. Artifacts the paper embeds

Three charts, each carrying one message, saved into `paper/img/` so the deployed page can use them with
relative paths.

In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 150, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

# --- chart 1: the signal the rule leans on, out of time ---
bands = pd.cut(test.momentum, bins=[-np.inf, 0.5, 0.8, 1.0, 1.2, np.inf],
               labels=["<0.5", "0.5-0.8", "0.8-1.0", "1.0-1.2", ">=1.2"])
band_tab = (test.assign(band=bands).dropna(subset=["band"])
            .groupby("band", observed=True).label.agg(["size", "mean"]).reset_index())
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(band_tab.band.astype(str), band_tab["mean"] * 100, color="#3b6ea5")
ax.axhline(test.label.mean() * 100, color="#c0392b", linestyle="--", linewidth=1.2,
           label=f"all pages: {test.label.mean() * 100:.0f}%")
for bar, (_, row) in zip(bars, band_tab.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5, f"n={int(row['size']):,}",
            ha="center", fontsize=8)
ax.set_xlabel("momentum in May  (2nd half ÷ 1st half)")
ax.set_ylabel("share declining ≥20% in June")
ax.set_title("In-month momentum, measured in May, predicts June decline")
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig(FIG / "signal-momentum.png")
plt.close(fig)

# --- chart 2: model vs baseline by K, on the future month ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve.k, curve.rule * 100, marker="o", label="rule (ML-07 baseline)", color="#7f8c8d")
ax.plot(curve.k, curve.logit * 100, marker="o", label="logistic regression", color="#3b6ea5")
ax.plot(curve.k, curve["logit x log(imp)"] * 100, marker="o", label="logit x log(imp) (deployed)", color="#e67e22")
ax.axhline(test_gated.label.mean() * 100, color="#c0392b", linestyle="--", linewidth=1.2,
           label=f"gated base rate: {test_gated.label.mean() * 100:.0f}%")
ax.set_xscale("log")
ax.set_xticks(curve.k, [str(k) for k in curve.k])
ax.set_xlabel("K — editor slots")
ax.set_ylabel("precision@K (%)")
ax.set_title("May → June, never-seen month: precision by queue depth")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "model-vs-baseline.png")
plt.close(fig)

# --- chart 3: grouped vs random — what memorisation looks like ---
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(selection))
ax.bar(x - 0.2, selection.p50_grouped * 100, width=0.4, label="grouped by client (honest)", color="#3b6ea5")
ax.bar(x + 0.2, selection.p50_random * 100, width=0.4, label="random split", color="#bdc3c7")
ax.set_xticks(x, selection.index)
ax.set_ylabel("precision@50 (%)")
ax.set_ylim(0, 110)
ax.set_title("Same rows, two splits: the flexible models lose the most")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "grouped-vs-random.png")
plt.close(fig)

for p in sorted(FIG.glob("*.png")):
    print(f"  {p}  {p.stat().st_size / 1024:.0f} KB")

  paper\img\grouped-vs-random.png  29 KB
  paper\img\model-vs-baseline.png  55 KB
  paper\img\signal-momentum.png  41 KB


In [11]:
metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "release": BUILD,
    "windows": {"train": {"features": TRAIN_FEATURES, "label": TRAIN_LABEL},
                "test": {"features": TEST_FEATURES, "label": TEST_LABEL}},
    "population": {"gate_impressions": GATE,
                   "train_items": int(len(train)), "train_clients": int(train.client_hash_id.nunique()),
                   "train_base_rate": round(float(train.label.mean()), 4),
                   "test_items": int(len(test)), "test_clients": int(test.client_hash_id.nunique()),
                   "test_base_rate": round(float(test.label.mean()), 4),
                   "test_gated_items": int(len(test_gated)),
                   "test_gated_base_rate": round(float(test_gated.label.mean()), 4)},
    "features": FEATURES,
    "label": "label = 1 when next month's impressions < 0.8 x feature month's",
    "selection": {"design": "5 x 20% held-out, grouped by client vs random, on the training month",
                  "precision_at_50": selection["p50_grouped"].round(3).to_dict(),
                  "precision_at_50_random": selection["p50_random"].round(3).to_dict(),
                  "auc_grouped": selection["auc_grouped"].round(3).to_dict()},
    "future_test": {"design": "fit 2026-03, predict 2026-05, score against 2026-06 (single use)",
                    "precision_at_50_gated": future["p@50_gated"].round(3).to_dict(),
                    "auc_gated": future["auc_gated"].round(3).to_dict(),
                    "precision_at_50_all": future["p@50_all"].round(3).to_dict()},
    "playbook": {"score": "P(decline) x log(1 + impressions)",
                 "rows_written": int(len(queue)), "clients": int(queue.client_hash_id.nunique()),
                 "top10_hits": int(queue.head(10).label.sum()),
                 "top50_hits": int(queue.label.sum()),
                 "top10": [{"rank": int(r.rank), "page": r.content_hash_id[:14],
                            "impressions": int(r.imp), "momentum": round(float(r.momentum), 3),
                            "position": round(float(r.pos), 1), "action": r.action,
                            "june_pct": round(float((r.imp_next - r.imp) / r.imp * 100), 1)}
                           for r in queue.head(10).itertuples()]},
    "seed": SEED,
    "generated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "notebook": "work/notebooks/capstone.ipynb",
}
(OUT / "capstone_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
print(f"receipt written: {OUT / 'capstone_metrics.json'}")
print(json.dumps(metrics["future_test"]["precision_at_50_gated"], indent=2))

receipt written: work\outputs\capstone_metrics.json
{
  "rule": 0.94,
  "logit": 0.98,
  "rf": 0.88,
  "gbm": 0.9
}


## 8. Reproducibility

- **Seeds:** `SEED = 42` for every split, forest and bootstrap; logistic regression is deterministic under lbfgs.
- **Run it:** clone the repo, `pip install -r requirements.txt`, then run this notebook top to bottom.
  It needs a Hugging Face READ token in `.env` (gitignored) or a Colab Secret `HF_TOKEN`; access to the
  gated release is instant. The first run scans four month partitions (~2–4 minutes) and caches the two
  frames under `work/outputs/` — later runs load the cache.
- **Environment:** DuckDB ≥ 1.0 (reads `hf://`), scikit-learn ≥ 1.4, pandas ≥ 2.2, matplotlib ≥ 3.8.
  Tree-ensemble numbers can move by a point or two across scikit-learn versions; the logistic model and
  the rule are stable to the third decimal.
- **Receipts:** `work/outputs/capstone_metrics.json` (committed) reproduces every number quoted in the
  paper; `work/outputs/capstone_action_queue.csv` is regenerated on each run and stays out of git.
- **The sealed month:** 2026-06 was unread until this notebook, then used once as the test label. There
  is no second holdout — that is the honest cost of a single snapshot, and it is why section 5 is longer
  than the results section.

## 9. Acknowledgments and data credit

Built on the **FlyRank ML Internship** dataset — <https://flyrank.ai>. The release is a pseudonymized,
gated warehouse snapshot; no client names, domains, raw URLs, or raw queries appear in this notebook or
in the deployed paper.

## Self-check

- [x] Every section filled — markdown reasoning **and** the code that backs it
- [x] Runs top to bottom with no errors (`Runtime → Run all`; executed locally with `nbclient`, outputs saved)
- [x] Baseline in every comparison table, computed in this run on the same rows and K
- [x] Base rates printed next to every precision number
- [x] Grouped-by-client validation, random split reported beside it, gap explained
- [x] Leakage checks asserted, not asserted-in-prose — including a feature matrix rebuilt without outcomes
- [x] Future-window test on a month never used for fitting, with a single use of the sealed month
- [x] Error cases and queue composition shown, not just the headline metric
- [x] Claims use observed / measured / directional / decision-support; no causal language
- [x] No client names, URLs, private queries, tokens, or raw exports
- [x] Metrics receipt committed: `work/outputs/capstone_metrics.json`
- [x] Charts written to `paper/img/` for the deployed page, relative paths only
- [x] Paper URL recorded in `submission/paper_url.txt`; all nine paper sections present, with the
      <https://flyrank.ai> data credit

## ML-12 — demo outline, social cut, employer summary

**5-minute demo outline**

1. *(0:00)* The decision: 50 editorial slots a week, several clients, one wrong page costs a day. Show the
   playbook queue, unsorted, as an editor sees it.
2. *(0:45)* The data: 79M-row warehouse release, two month-pairs, and the three things excluded —
   the query table (window overlaps the label), current-state metadata, and the staleness field that only
   exists as a 2026-07 snapshot.
3. *(1:30)* The baseline first: a three-condition rule from week 4 reaches 47/50 on a future month. That is
   the number every model has to beat.
4. *(2:30)* The split: show the grouped-vs-random chart. Boosting scores 0.984 on a random split and 0.836
   when clients are held out; the forest collapses from 0.968 to 0.700. The winner is the boring logistic
   regression, because it did not memorise clients.
5. *(3:30)* The result: 49/50 against 47/50 on May→June, a two-page difference with overlapping intervals —
   and a queue that spans 20 clients instead of 11.
6. *(4:30)* The playbook: `check_indexing` for structural breaks, `refresh_content` for visible slips, and
   the one number that would settle causality — a four-week comparison after refreshes.

**Social-post cut**

> I ranked 60k pages for a content editor's 50 weekly refresh slots. A hand-written rule got 47/50 on a
> month it never saw; a logistic model got 49/50. The interesting part: the models that *looked* best were
> memorising clients — gradient boosting fell from 98% to 84% the moment I grouped the split by client.
> Built on the FlyRank ML Internship dataset.

**Employer-facing 3 sentences**

I built a refresh-prioritisation system on 79M rows of real search-performance data: monthly features,
a future-window label, a transparent baseline and a validated model, evaluated on a month never used for
fitting. The work's value is the audit trail — grouped validation, a frozen baseline, leakage checks and a
backtested queue — not the model, which improves on the rule by two pages at the editor's operating point.
Along the way it produced two findings worth keeping: staleness cannot be measured honestly in this
snapshot, and every flexible model's apparent edge was largely client memorisation.